<a href="https://colab.research.google.com/github/vedikakapoor27/mini_gpt/blob/main/mini_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"vedika2704","key":"e666cde1f74a2cbcc26a09dd8a9ea120"}'}

In [8]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

mkdire here is creating a hidden folder (.kaggle)
Kaggle needs this folder to store your API key

!cp
Copies your uploaded kaggle.json into that folder

 Why?
So Kaggle can authenticate (log in)

Changes file permissions

600 = only YOU can read/write it

 Why?
Kaggle requires this for security
(otherwise it throws error)

In [2]:
!kaggle datasets download -d kingburrito666/shakespeare-plays
!unzip shakespeare-plays.zip


Dataset URL: https://www.kaggle.com/datasets/kingburrito666/shakespeare-plays
License(s): unknown
100% 4.55M/4.55M [00:01<00:00, 4.33MB/s]

Archive:  shakespeare-plays.zip
  inflating: Shakespeare_data.csv    
  inflating: alllines.txt            
  inflating: william-shakespeare-black-silhouette.jpg  


-d means dataset so it downloads the dataset and extracts all the data

In [21]:
import os
print(os.listdir())


['.config', 'alllines.txt', 'shakespeare-plays.zip', 'william-shakespeare-black-silhouette.jpg', 'kaggle.json', 'Shakespeare_data.csv', 'sample_data']


this will show all the files in current folder
import os is used to bring in Python’s built-in OS (Operating System) module


alllines.txt → FULL TEXT DATA (you will use this)
Shakespeare_data.csv → structured data (not needed for GPT here)
kaggle.json → your login key
sample_data → default Colab folder

In [22]:
#read it in to inspect it
with open('alllines.txt','r',encoding='utf-8') as f:
  text=f.read()



In [23]:
print("length of datasets in characters:",len(text))

length of datasets in characters: 4583798


In [24]:
print(text[:993])

"ACT I"
"SCENE I. London. The palace."
"Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others"
"So shaken as we are, so wan with care,"
"Find we a time for frighted peace to pant,"
"And breathe short-winded accents of new broils"
"To be commenced in strands afar remote."
"No more the thirsty entrance of this soil"
"Shall daub her lips with her own children's blood,"
"Nor more shall trenching war channel her fields,"
"Nor bruise her flowerets with the armed hoofs"
"Of hostile paces: those opposed eyes,"
"Which, like the meteors of a troubled heaven,"
"All of one nature, of one substance bred,"
"Did lately meet in the intestine shock"
"And furious close of civil butchery"
"Shall now, in mutual well-beseeming ranks,"
"March all one way and be no more opposed"
"Against acquaintance, kindred and allies:"
"The edge of war, like an ill-sheathed knife,"
"No more shall cut his master. Therefore, friends,"
"As far as to the sepulchre of Christ,"



In [6]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print('' .join(chars))
print(vocab_size)

	
 !"$'(),-.0123456789:?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz
78


set(text) → gets unique characters (removes duplicates)
list(...) → converts them into a list
sorted(...) → arranges characters in order

vocab_size = len(chars)
👉 counts total number of unique characters

print(''.join(chars))
👉 shows all characters together (for checking)
print(vocab_size)
👉 shows how many characters exist

The model needs:

all possible characters
total number of characters

👉 to convert text into numbers and make predictions

In [25]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s]# encoder: take a string, output a list of integers
decode=lambda l: ''.join([itos[i] for i in l])# decoder: take a list of integers, output a stri
print(encode("hii there"))
print(decode(encode("hii there")))

[59, 60, 60, 2, 71, 59, 56, 69, 56]
hii there


# 📌 Mini GPT – Data Preprocessing & Tokenization Summary

## 🔹 What this part does

This step prepares raw text data so that the model can understand it.
Since neural networks cannot process text directly, we convert characters into numbers.

---

## 🔹 Step-by-step explanation

### 1. Create vocabulary (unique characters)

We first extract all unique characters from the dataset.

```python
chars = sorted(list(set(text)))
vocab_size = len(chars)
```

👉 `chars` → list of unique characters
👉 `vocab_size` → total number of unique characters

---

### 2. Create mappings

```python
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
```

👉 `stoi` (string → integer)

* Converts characters to numbers
* Example: `'a' → 0`

👉 `itos` (integer → string)

* Converts numbers back to characters
* Example: `0 → 'a'`

---

### 3. Encoding (text → numbers)

```python
encode = lambda s: [stoi[c] for c in s]
```

👉 Converts a string into a list of integers
Example:

```python
encode("hi")
# Output: [7, 8] (depends on mapping)
```

---

### 4. Decoding (numbers → text)

```python
decode = lambda l: ''.join([itos[i] for i in l])
```

👉 Converts numbers back into readable text
Example:

```python
decode([7, 8])
# Output: "hi"
```

---

### 5. Test encoding & decoding

```python
print(encode("hii there"))
print(decode(encode("hii there")))
```

👉 Ensures both functions work correctly

---

## 🔹 Why this is needed

* Models cannot understand text directly ❌
* They only understand numbers ✅
* So we convert text → numbers before training

---

## 🔹 Important concept

This step is called:
👉 **Character-level Tokenization**

---

## 🔹 Where this fits in pipeline

1. Load dataset
2. Tokenization (this step) ✅
3. Convert to tensor
4. Create batches
5. Train model

---

## 🔹 Final takeaway

👉 Tokenization = preprocessing
👉 Training = learning patterns

---

## 🔹 Interview-ready line

“I implemented character-level tokenization by mapping each unique character to an integer index and used it to prepare data for training a GPT-style model.”

---


In [26]:
import torch
data=torch.tensor(encode(text) , dtype=torch.long)
print(data.shape,data.dtype)
print(data[:993])

torch.Size([4583798]) torch.int64
tensor([ 4, 24, 26, 43,  2, 32,  4,  1,  4, 42, 26, 28, 37, 28,  2, 32, 11,  2,
        35, 66, 65, 55, 66, 65, 11,  2, 43, 59, 56,  2, 67, 52, 63, 52, 54, 56,
        11,  4,  1,  4, 28, 65, 71, 56, 69,  2, 34, 32, 37, 30,  2, 31, 28, 37,
        41, 48,  9,  2, 35, 38, 41, 27,  2, 33, 38, 31, 37,  2, 38, 29,  2, 35,
        24, 37, 26, 24, 42, 43, 28, 41,  9,  2, 71, 59, 56,  2, 28, 24, 41, 35,
         2, 66, 57,  2, 46, 28, 42, 43, 36, 38, 41, 28, 35, 24, 37, 27,  9,  2,
        42, 32, 41,  2, 46, 24, 35, 43, 28, 41,  2, 25, 35, 44, 37, 43,  9,  2,
        52, 65, 55,  2, 66, 71, 59, 56, 69, 70,  4,  1,  4, 42, 66,  2, 70, 59,
        52, 62, 56, 65,  2, 52, 70,  2, 74, 56,  2, 52, 69, 56,  9,  2, 70, 66,
         2, 74, 52, 65,  2, 74, 60, 71, 59,  2, 54, 52, 69, 56,  9,  4,  1,  4,
        29, 60, 65, 55,  2, 74, 56,  2, 52,  2, 71, 60, 64, 56,  2, 57, 66, 69,
         2, 57, 69, 60, 58, 59, 71, 56, 55,  2, 67, 56, 52, 54, 56,  2, 71, 66,
      

. Import PyTorch
import torch

👉 Imports PyTorch library used for building and training models.

2. Convert text → tensor
data = torch.tensor(encode(text), dtype=torch.long)

👉 Step breakdown:

encode(text) → converts text into list of integers
torch.tensor(...) → converts list into tensor
dtype=torch.long → ensures integers (required for embeddings)

✔ Output: numerical representation of full dataset

3. Check shape & datatype
print(data.shape, data.dtype)

👉 Shows:

shape → total number of characters
dtype → integer type (int64)
4. View sample data
print(data[:1000])

👉 Displays first 1000 elements of dataset
✔ Helps verify encoding worked correctly

🔹 Key concept

👉 Text → Numbers → Tensor

Model understands only numbers, not text.

🔹 Why this is important
Neural networks work on tensors
This step prepares data for training
Required before batching and model input
🔹 One-line summary

Converts tokenized text into a PyTorch tensor so it can be used as input for training the GPT model.

In [27]:
n=int(0.9*len(data))# Let's now split up the data into train and validation sets
train_data=data[:n]
val_data=data[n:]

In [28]:
block_size=8
train_data[:block_size+1]

tensor([ 4, 24, 26, 43,  2, 32,  4,  1,  4])

Split dataset
train_data → 90% (used for learning)
val_data → 10% (used for evaluation)

👉 Helps prevent overfitting and check model performance.

2. Set block size
block_size = 8
👉 Model looks at 8 previous tokens (context)
3. Take sample sequence
train_data[:block_size+1] → 9 tokens

👉 Why 9?

First 8 → input (x)
Last 1 → target (y)
🔹 Key idea

We prepare data so the model learns:
👉 “Given previous tokens, predict the next token”

In [29]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([4]) the target: 24
when input is tensor([ 4, 24]) the target: 26
when input is tensor([ 4, 24, 26]) the target: 43
when input is tensor([ 4, 24, 26, 43]) the target: 2
when input is tensor([ 4, 24, 26, 43,  2]) the target: 32
when input is tensor([ 4, 24, 26, 43,  2, 32]) the target: 4
when input is tensor([ 4, 24, 26, 43,  2, 32,  4]) the target: 1
when input is tensor([ 4, 24, 26, 43,  2, 32,  4,  1]) the target: 4


What this does
1. Create input (x)
x = train_data[:block_size]
👉 Takes first block_size tokens
👉 This is the context (input to model)
2. Create target (y)
y = train_data[1:block_size+1]
👉 Same sequence but shifted by 1
👉 Represents the next token to predict
3. Loop through sequence
for t in range(block_size):

👉 Iterates over each position

4. Build context
context = x[:t+1]

👉 Takes increasing portion of input
Example:

[4]
[4, 24]
[4, 24, 26]
5. Assign target
target = y[t]

👉 Next token corresponding to that context

6. Print pairs
print(...)

👉 Shows how model will learn:

Example:

Input [4] → Target 24
Input [4, 24] → Target 26
🔥 Key idea

👉 Model learns:
“Given previous tokens, predict the next token”

🔹 Why this is important
Converts raw data into training format
This is the core learning mechanism of GPT
🔹 One-line summary

Creates multiple (context → next token) training examples from a sequence to train the model for next-token prediction.

In [30]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")



inputs:
torch.Size([4, 8])
tensor([[69, 52, 54, 60, 66, 72, 70,  2],
        [60, 70,  2, 59, 60, 70,  2, 57],
        [ 4,  1,  4, 67, 76, 69, 52, 64],
        [56, 71,  9,  2, 71, 66,  2, 70]])
targets:
torch.Size([4, 8])
tensor([[52, 54, 60, 66, 72, 70,  2, 64],
        [70,  2, 59, 60, 70,  2, 57, 52],
        [ 1,  4, 67, 76, 69, 52, 64, 60],
        [71,  9,  2, 71, 66,  2, 70, 66]])
----
when input is [69] the target: 52
when input is [69, 52] the target: 54
when input is [69, 52, 54] the target: 60
when input is [69, 52, 54, 60] the target: 66
when input is [69, 52, 54, 60, 66] the target: 72
when input is [69, 52, 54, 60, 66, 72] the target: 70
when input is [69, 52, 54, 60, 66, 72, 70] the target: 2
when input is [69, 52, 54, 60, 66, 72, 70, 2] the target: 64
when input is [60] the target: 70
when input is [60, 70] the target: 2
when input is [60, 70, 2] the target: 59
when input is [60, 70, 2, 59] the target: 60
when input is [60, 70, 2, 59, 60] the target: 70
when input is 

What this does
1. Set parameters
batch_size = 4 → number of sequences processed together
block_size = 8 → length of each sequence (context)
2. Random starting points
ix = torch.randint(len(data) - block_size, (batch_size,))

👉 Picks random indices to sample different parts of data
👉 Ensures model sees varied data every time

3. Create input batch (x)
x = torch.stack([data[i:i+block_size] for i in ix])

👉 Creates multiple sequences
👉 Shape: (batch_size, block_size)

4. Create target batch (y)
y = torch.stack([data[i+1:i+block_size+1] for i in ix])

👉 Same sequences shifted by 1
👉 Targets = next tokens

🔹 Example output
xb.shape → (4, 8)
yb.shape → (4, 8)

👉 4 sequences, each of length 8

🔹 Loop explanation
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]

👉 For each sequence and each position:

context → input tokens so far
target → next token to predict
🔥 Key idea

👉 Model learns:
Given context → predict next token

🔹 Why batching?
Faster training
Parallel processing
Better learning efficiency

In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 78])
tensor(4.8167, grad_fn=<NllLossBackward0>)
	z,amn?PM	ntzyFVJ[XNcvMTUsknBlOrsbxLU!S.MLA$Yhed'GC'XRIpL'8ysTx( u	4Y7lMUZWN"zj".[X48EGs3OsbqCc1MXe$e


In [32]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [33]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.7004714012146


In [34]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

	7 1Rms[]:Ut)RVuk,]DnTueueKNccQlmH19RqXh")kxTO)Ye?3S8[]'VxKucR?:lqm2ov[8BY:K33sV5R(s5TdhzZ3YCmV]x-xUTr$[,H!89.Eeue:jUZQx
cG9tkjna
 TbN-l
R5T.EBvOD)[Qk].'L[ipJdMD-50.qBp1]Qkn)jt]C3b)	Eyots?ooH8x	"vrT	emngz6k:WVuB,99y)$N	c,1
-xt
d(Jp8"MAkLZHm(Nuj!'L:b	7(kqD[
"M7bccD"Tkp eiz80fb	e.8"F,T[ze$CrSn7uNcm
fT[K	[q?i?
d.7Dw!B-8HZpxfX4HVS7dr:h'Mov.EstytP$H$HRSymKxCKlmjB a6 L.
,0kncR6.mI$HmObf6Wy)fwLz-0Wjw[iEyGaR6
SMCZvp.yeG[r'Vtr5CLdGS7KkIP9IR"c-D-RIKfblG-s	DCVWjL0$H 1ahOb1 ]rZ Q,'"RS7KdFUVLO02[BY8HzALOyUiuy


In [35]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [36]:
torch.manual_seed(1337)
B,T,C=4,8,2
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [37]:
xbow=torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev=x[b,:t+1]
    xbow[b,t]=torch.mean(xprev,0)



In [20]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)


In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape